In [1]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp
from matplotlib import pyplot as plt
from pennylane.operation import Operation, AnyWires
import os
import pandas as pd
from scipy.stats import uniform_direction


num_qubits = 5

# Initialize the device
dev = qml.device("lightning.qubit", wires=num_qubits)

In [2]:
# Construct the Hamiltonian terms
Hamiltonian_terms = []

# Interaction terms: XiX(i+1) + YiY(i+1) + ZiZ(i+1)
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * (qml.PauliX(i) @ qml.PauliX((i+1)%num_qubits)) +
                                    (qml.PauliY(i) @ qml.PauliY((i+1)%num_qubits)) 
                                    + (qml.PauliZ(i) @ qml.PauliZ((i+1)%num_qubits)))

# Magnetic field terms: hZi
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * qml.PauliZ(i))

# Define the Hamiltonian
Hamiltonian = qml.Hamiltonian(coeffs=[1] * len(Hamiltonian_terms), observables=Hamiltonian_terms)

In [3]:
# define identity matrix and pauli matrices

Id = np.matrix([[1,0],
               [0,1]])

x = np.matrix([[0,1],
               [1,0]])

y = np.matrix([[0,-1j],
               [1j,0]])

z = np.matrix([[1,0],
               [0,-1]])

In [4]:


def entangling_layer_Ansatz_A(num_qubits):
    m = 0
    n = 1
    while m+1 < num_qubits:
        qml.CZ(wires=[m,m+1])
        m+=2
    
    while n+1 < num_qubits:
        qml.CZ(wires=[n,n+1])
        n+=2

@qml.qnode(dev)
def circuit(n_vectors, num_layers):
    """Parameterized quantum circuit with free-axis rotations"""
    
    for j in range(num_layers):
        for k in range(num_qubits):
            n1, n2, n3 = n_vectors[k + num_qubits * j]
            unitary = -1j * (x * n1 + y* n2 + z*n3)
            
            qml.QubitUnitary(unitary, wires = k)
    
        entangling_layer_Ansatz_A(num_qubits)

    return qml.expval(Hamiltonian)

@qml.qnode(dev)
def circuit_state(n_vectors, num_layers, d, gate_type):
    """Parameterized quantum circuit with free-axis rotations"""
    ind = 0
    for j in range(num_layers):
        for k in range(num_qubits):
            if ind == d:
                if gate_type == "X":
                    qml.QubitUnitary(x, wires = k)

                elif gate_type == "Y":
                    qml.QubitUnitary(y, wires = k)

                elif gate_type == "Z":
                    qml.QubitUnitary(z, wires = k)

                elif gate_type == "XY":
                    unitary = (x + y) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
   
                elif gate_type == "XZ":
                    unitary = (x + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

                elif gate_type == "YZ":
                    unitary = (y + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

            else:
                n1, n2, n3 = n_vectors[k + num_qubits * j]
                unitary = -1j * (x * n1 + y* n2 + z*n3)
                
                qml.QubitUnitary(unitary, wires = k)

            ind += 1
    
        entangling_layer_Ansatz_A(num_qubits)

    
    return qml.expval(Hamiltonian)


In [5]:

def compute_Rd_matrix(n_vectors, num_layers, d):
    """Compute the Rd matrix for a specific gate d"""

    rx = circuit_state(n_vectors, num_layers, d, gate_type="X")
    ry = circuit_state(n_vectors, num_layers, d, gate_type="Y")
    rz = circuit_state(n_vectors, num_layers, d, gate_type="Z")
    rxy = circuit_state(n_vectors, num_layers, d, gate_type="XY")
    rxz = circuit_state(n_vectors, num_layers, d, gate_type="XZ")
    ryz = circuit_state(n_vectors, num_layers, d, gate_type="YZ")
    
    Rd=[[2*rx,        2*rxy-rx-ry, 2*rxz-rx-rz],
        [2*rxy-rx-ry,        2*ry, 2*ryz-ry-rz],
        [2*rxz-rx-rz, 2*ryz-ry-rz,        2*rz]]

    return Rd



In [6]:


def fraxis_optimization(n_vectors, num_layers, iters, freeze_threshold, freeze_iters_k):
    """Implement the Fraxis algorithm"""
    num_gates = len(n_vectors)
    vals = []
    dists = []

    freeze_counters = np.zeros(len(n_vectors))    
    
    gate_opts_tresh = num_qubits * num_layers * iters 
    gate_opts = 0

    while True:
        
        if gate_opts > gate_opts_tresh:
            break
        
        for d in range(num_gates):

            if gate_opts > gate_opts_tresh:
                break
            
            if freeze_counters[d] > 0:
                freeze_counters[d] = freeze_counters[d] - 1
                #print(d)
                continue

            prev_axis = np.array(n_vectors[d].copy())

            current_val = circuit(n_vectors, num_layers)
            vals.append(current_val)
            Rd = compute_Rd_matrix(n_vectors, num_layers, d)        

            eigVal, eigVec = np.linalg.eig(Rd)
            eigVec = np.transpose(eigVec)

            sid = np.argmin(eigVal)

            expected_val = np.amin(eigVal)*0.5

            new_vec = [eigVec[sid][0], eigVec[sid][1], eigVec[sid][2]] 
            
            if expected_val < current_val:
                n_vectors[d] = new_vec


            current_axis = np.array(n_vectors[d].copy())

            gc_dist = np.arccos(np.dot(prev_axis, current_axis))
            
            if gc_dist > np.pi/2.0:
                gc_dist = np.pi - gc_dist
            

            if (gc_dist < freeze_threshold): 
                freeze_counters[d] = freeze_iters_k
                #print("Freeze d,", d)
                
            gate_opts += 1

        

    return n_vectors, vals, dists



In [ ]:
# Initialize parameters and run optimization
layers = [5*num_qubits]
freeze_iters_k_list = [3]
dvals = [0.025, 0.01, 0.005]

iters = 10
trials = 20

uniform_sphere_dist = uniform_direction(3)


for num_layers in layers:
    for d in dvals:
        freeze_threshold = d

        for freeze_iters_k in freeze_iters_k_list:
            for trial in range(trials):
                print("trials", trial+1)
                num_gates = num_qubits * num_layers
                n_vectors = uniform_sphere_dist.rvs(num_gates)
                
                optimal_n_vectors, opt_vals, dists = fraxis_optimization(n_vectors, num_layers, iters, freeze_threshold,freeze_iters_k)
                                
                data_file = f"1DHeisenberg_{num_qubits}Q_Fraxis_GateFreeze_d{d}_FreezeIter{freeze_iters_k}_{iters}cycles_{num_layers}layers_{trials}trials_A.xlsx"   
                
                if not os.path.exists(data_file):
                    df2 = pd.DataFrame()
                    df2.to_excel(data_file)
        
                df2 = pd.read_excel(data_file)
        
                if len(df2.columns) < trials:
                    
                    df2[f"col{len(df2.columns)}"] = pd.Series(opt_vals)
                    df2.to_excel(data_file,index = False)
                else:
                    break
    